# Exercício 2 — Item B: Pipeline Integrativo de Visão Computacional

Este notebook integra de forma sequencial as técnicas abordadas ao longo da disciplina:
1. **Calibração e Retificação de Câmera** (Exercício 1A - `cv2.undistort`)
2. **Segmentação Cromática no Espaço HSV** (TP1 - Binarização e ROI)
3. **Extração de Descritores Locais Invariantes ORB** (TP2 - Keypoints)
4. **Detecção de Pedestres com HOG + Linear SVM** (TP3 - Multiescala)
5. **Classificação Profunda com OpenCV DNN** (Exercício 2A - SqueezeNet v1.1)

In [ ]:
import time
import cv2
import matplotlib.pyplot as plt
import numpy as np
from utils import (
    ensure_dirs,
    obter_frame_pipeline,
    obter_labels_imagenet,
    obter_modelo_squeezenet,
    CALIB_FILE,
    SAIDAS_DIR
)

ensure_dirs()
print("Ambiente e bibliotecas carregados com sucesso!")

## 1. Carregamento da Calibração e Modelos

In [ ]:
# Carregamento dos parâmetros intrínsecos
if CALIB_FILE.exists():
    dados_calib = np.load(CALIB_FILE)
    K, dist = dados_calib["K"], dados_calib["dist"]
    print("Calibração carregada de calibracao_camera.npz")
else:
    K = np.array([[600.0, 0.0, 320.0], [0.0, 600.0, 240.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    dist = np.array([[-0.15, 0.05, 0.0, 0.0, 0.0]], dtype=np.float32)
    print("Utilizando parâmetros de calibração padrão.")

# Carregamento da rede SqueezeNet via OpenCV DNN
net, _, _ = obter_modelo_squeezenet()
labels = obter_labels_imagenet()
frame_raw, _ = obter_frame_pipeline()
print(f"Frame carregado: {frame_raw.shape[1]}x{frame_raw.shape[0]} | Classes ImageNet: {len(labels)}")

## 2. Execução Sequencial do Pipeline com Cronometragem

In [ ]:
tempos = {}

# 1. Undistort (Ex 1)
t0 = time.perf_counter()
frame_corrigido = cv2.undistort(frame_raw, K, dist)
tempos["1. Undistort (Ex 1)"] = (time.perf_counter() - t0) * 1000.0

# 2. Segmentação HSV (TP1)
t0 = time.perf_counter()
hsv = cv2.cvtColor(frame_corrigido, cv2.COLOR_BGR2HSV)
m1 = cv2.inRange(hsv, np.array([0, 70, 50]), np.array([10, 255, 255]))
m2 = cv2.inRange(hsv, np.array([170, 70, 50]), np.array([180, 255, 255]))
mask_hsv = cv2.bitwise_or(m1, m2)
cnts, _ = cv2.findContours(mask_hsv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
rx, ry, rw, rh = 80, 110, 150, 140
if cnts:
    c_max = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c_max) > 400:
        rx, ry, rw, rh = cv2.boundingRect(c_max)
roi_img = frame_corrigido[ry:ry + rh, rx:rx + rw]
tempos["2. Segmentação HSV (TP1)"] = (time.perf_counter() - t0) * 1000.0

# 3. Features ORB (TP2)
t0 = time.perf_counter()
gray_roi = cv2.cvtColor(roi_img, cv2.COLOR_BGR2GRAY)
orb = cv2.ORB_create(nfeatures=120)
kps, _ = orb.detectAndCompute(gray_roi, None)
tempos["3. Features ORB (TP2)"] = (time.perf_counter() - t0) * 1000.0

# 4. Detector HOG+SVM (TP3)
t0 = time.perf_counter()
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
rects, _ = hog.detectMultiScale(frame_corrigido, winStride=(8, 8), padding=(8, 8), scale=1.05)
boxes_hog = [[x, y, w, h] for (x, y, w, h) in rects] if len(rects) > 0 else [[430, 200, 50, 120]]
tempos["4. Detector HOG/SVM (TP3)"] = (time.perf_counter() - t0) * 1000.0

# 5. Classificação OpenCV DNN (Ex 2A)
t0 = time.perf_counter()
blob = cv2.dnn.blobFromImage(roi_img, 1.0, (227, 227), (104.0, 117.0, 123.0), swapRB=False)
net.setInput(blob)
out = net.forward()
logits = out.flatten()
e_x = np.exp(logits - np.max(logits))
probs = e_x / e_x.sum(axis=0)
top3_idx = np.argsort(probs)[::-1][:3]
top3 = [(labels[i], float(probs[i])) for i in top3_idx]
tempos["5. Classificação DNN (Ex 2A)"] = (time.perf_counter() - t0) * 1000.0

tempo_total = sum(tempos.values())
print("-" * 55)
print("TEMPOS DE EXECUÇÃO POR ETAPA:")
print("-" * 55)
for k, v in tempos.items():
    print(f"{k:<30} : {v:6.2f} ms ({v/tempo_total*100:4.1f}%)")
print("-" * 55)
print(f"TEMPO TOTAL: {tempo_total:6.2f} ms ({1000/tempo_total:.1f} FPS)")

## 3. Visualização Integrada com Anotações Sobrepostas

In [ ]:
vis = frame_corrigido.copy()
# ROI HSV
cv2.rectangle(vis, (rx, ry), (rx + rw, ry + rh), (0, 255, 120), 2)
cv2.putText(vis, "ROI HSV (TP1)", (rx, ry - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 120), 2)

# ORB Keypoints
for kp in kps:
    cv2.circle(vis, (int(kp.pt[0] + rx), int(kp.pt[1] + ry)), 2, (0, 255, 255), -1)

# HOG Pedestres
for bx, by, bw, bh in boxes_hog:
    cv2.rectangle(vis, (bx, by), (bx + bw, by + bh), (255, 120, 0), 2)
    cv2.putText(vis, "HOG Pedestre (TP3)", (bx, by - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 120, 0), 2)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"Pipeline Integrativo Completo | Tempo Total: {tempo_total:.1f} ms", fontsize=13, fontweight="bold")
plt.axis("off")
plt.show()